# Import Required Libraries
Import the necessary libraries, including h5py and numpy.

In [1]:
# Import Required Libraries
import pandas as pd
import h5py
import numpy as np
import nibabel as nib
# h5py is used for handling HDF5 files
# numpy is used for numerical operations

# Load h5 File
Use h5py to load an h5 file from a specified path.

In [6]:
# Load h5 File
file_path = 'data/BraTS2020_training_data/content/data/volume_1_slice_0.h5'  # specify the path to your h5 file

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Explore the structure of the H5 file
    def print_structure(name, obj):
        print(name)
    h5_file.visititems(print_structure)
    
    # Access the 'image' dataset
    if 'image' in h5_file:
        image_data = h5_file['image']
        print("Image shape:", image_data.shape)
    else:
        print("Dataset 'image' not found in the file.")
    
    # Access the 'mask' dataset
    if 'mask' in h5_file:
        mask_data = h5_file['mask']
        print("Mask shape:", mask_data.shape)
    else:
        print("Dataset 'mask' not found in the file.")

image
mask
Image shape: (240, 240, 4)
Mask shape: (240, 240, 3)


In [2]:
import os
from PIL import Image
import numpy as np
# Directory containing the images
main_dir = 'data/NINS_Dataset/'
image_dir = 'Brain Atrophy'

# Initialize a list to store the images
image_list = []

# Iterate over all files in the directory
for filename in os.listdir(main_dir + image_dir):
    if filename.endswith('.png') or filename.endswith('.jpg'):  # Adjust the file extensions as needed
        image_path = os.path.join(main_dir + image_dir, filename)
        image = Image.open(image_path)
        image = image.resize((240, 240), Image.ANTIALIAS)
        image = image.convert('L')
        image_array = np.array(image)
        
        image_list.append(image_array)

# Stack images along a new dimension
stacked_images = np.stack(image_list, axis=0)

print("Stacked images shape:", stacked_images.shape)
out_path = 'data/NINS/' + image_dir + '.nii'
image_nii = nib.Nifti1Image(stacked_images, np.eye(4))
nib.save(image_nii, out_path)

Stacked images shape: (264, 240, 240)


In [ ]:

# Load the CSV file
csv_path = 'data/BraTS2020_training_data/content/data/meta_data.csv'
df = pd.read_csv(csv_path)

# Group by volume to process each volume separately
grouped = df.groupby('volume')

for volume_id, group in grouped:
    # Initialize lists to store image and mask slices
    image_slices = []
    mask_slices = []

    for _, row in group.iterrows():
        h5_path = "data/BraTS2020_training_data" + row['slice_path']
        
        with h5py.File(h5_path, 'r') as h5_file:
            # Assuming the datasets are named 'image' and 'mask'
            image_slices.append(h5_file['image'][:])
            mask_slices.append(h5_file['mask'][:])
    # Stack slices to form 3D volumes
    image_slices = np.array(image_slices)  # shape (num_slices, h, w, 4)
    mask_slices = np.array(mask_slices)    # shape (num_slices, h, w, 3)
    
    # Transpose to get shape (h, w, num_slices, 4) for images and (h, w, num_slices, 3) for masks
    image_volumes = np.transpose(image_slices, (1, 2, 0, 3))
    mask_volumes = np.transpose(mask_slices, (1, 2, 0, 3))
    
    # Save each MRI type and mask type as separate NIfTI images
    mri_types = ['type1', 'type2', 'type3', 'type4']
    mask_types = ['mask1', 'mask2', 'mask3']
    
    for i, mri_type in enumerate(mri_types):
        image_nii = nib.Nifti1Image(image_volumes[..., i], np.eye(4))
        image_nii_path = f'data/Brats/volume_{volume_id}_{mri_type}.nii.gz'
        nib.save(image_nii, image_nii_path)
        print(f'Saved {image_nii_path}')
    
    for j, mask_type in enumerate(mask_types):
        mask_nii = nib.Nifti1Image(mask_volumes[..., j], np.eye(4))
        mask_nii_path = f'data/Brats/volume_{volume_id}_{mask_type}.nii.gz'
        nib.save(mask_nii, mask_nii_path)
        print(f'Saved {mask_nii_path}')


# Explore h5 File Structure
Explore the structure of the h5 file, including groups and datasets.

In [ ]:
# Explore h5 File Structure

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Function to recursively explore the structure of the h5 file
    def explore_h5_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")

    # Visit all items in the file
    h5_file.visititems(explore_h5_structure)

# Extract Data from h5 File
Extract specific datasets from the h5 file and convert them to numpy arrays for further analysis.

In [ ]:
# Extract Data from h5 File

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Extract specific datasets
    dataset_1 = h5_file['dataset_1'][:]  # replace 'dataset_1' with your actual dataset name
    dataset_2 = h5_file['dataset_2'][:]  # replace 'dataset_2' with your actual dataset name

# Convert datasets to numpy arrays
array_1 = np.array(dataset_1)
array_2 = np.array(dataset_2)

# Display the extracted data
print("Dataset 1:", array_1)
print("Dataset 2:", array_2)

In [2]:
import numpy as np

In [3]:
given_labels = np.load('data/labels_classes_priors/synthseg_segmentation_labels_2.0.npy')
label_names = np.load('data/labels_classes_priors/synthseg_segmentation_names_2.0.npy')
denoiser =  np.load('data/labels_classes_priors/synthseg_denoiser_labels_2.0.npy')
topological_classes = np.load('data/labels_classes_priors/synthseg_topological_classes_2.0.npy')
parcellation_labels = np.load('data/labels_classes_priors/synthseg_parcellation_labels.npy')
parcellation_names = np.load('data/labels_classes_priors/synthseg_parcellation_names.npy')
qc_labels = np.load('data/labels_classes_priors/synthseg_qc_labels.npy')
qc_names = np.load('data/labels_classes_priors/synthseg_qc_names.npy')

In [4]:

print( "denoiser", len(denoiser))
print("topological_classes", len(topological_classes))
print("given_labels", len(given_labels))
print("parcellation", len(parcellation_labels))
print("qc_labels", len(qc_labels))

denoiser 55
topological_classes 55
given_labels 55
parcellation 69
qc_labels 54


In [5]:
unique_num = 100
new_label = np.array([101,102,103])
new_names = np.array(['tumor-1', 'tumor-2', 'tumor-3'])
updated_label = np.append(given_labels, new_label)
updated_names = np.append(label_names, new_names)
new_denoiser = np.ones(len(new_label))
updated_denoiser = np.append(denoiser, new_denoiser)
new_topological_classes = np.ones(len(new_label))*unique_num
updated_topological_classes = np.append(topological_classes, new_topological_classes)

In [8]:
len(np.unique(updated_label))

36

In [36]:
import tensorflow as tf
import numpy as np
import h5py

In [40]:
import tensorflow as tf
from tensorflow.keras.models import load_model

In [41]:
model_path = "models/synthseg_2.0.h5"  # Replace with the actual path to your model

In [7]:
[None]*3 + [1]

[None, None, None, 1]

In [11]:
import os
import numpy as np
import tensorflow as tf
from ext.lab2im import layers
from ext.neuron import models as nrn_models

# Define the model path
model_path = "models/brats_synthseg_2.0.h5" # Replace with the actual path

# Define input parameters
input_shape = [None,None,None,1]  # 3D input with 1 channel
labels_segmentation = np.unique(updated_label)  # Example label list
n_levels = 5
nb_conv_per_level = 2
conv_size = 3
unet_feat_count = 24
feat_multiplier = 2
activation = 'elu'
sigma_smoothing = 0
flip_indices = None
gradients = False

# Define the build_model function
def build_model(path_model,
                input_shape,
                labels_segmentation,
                n_levels,
                nb_conv_per_level,
                conv_size,
                unet_feat_count,
                feat_multiplier,
                activation,
                sigma_smoothing,
                flip_indices,
                gradients):
    assert os.path.isfile(path_model), "The provided model path does not exist."

    # Get the number of labels
    n_labels_seg = len(labels_segmentation)

    # Build the UNet
    net = nrn_models.unet(input_shape=input_shape,
                          nb_labels=n_labels_seg,
                          nb_levels=n_levels,
                          nb_conv_per_level=nb_conv_per_level,
                          conv_size=conv_size,
                          nb_features=unet_feat_count,
                          feat_mult=feat_multiplier,
                          activation=activation,
                          batch_norm=-1)
    net.load_weights(path_model, by_name=True, skip_mismatch=True)

    # Smooth posteriors if specified
    if sigma_smoothing > 0:
        last_tensor = net.output
        last_tensor = layers.GaussianBlur(sigma=sigma_smoothing)(last_tensor)
        net = tf.keras.models.Model(inputs=net.inputs, outputs=last_tensor)

    return net

# Load the model
model = build_model(model_path,
                    input_shape,
                    labels_segmentation,
                    n_levels,
                    nb_conv_per_level,
                    conv_size,
                    unet_feat_count,
                    feat_multiplier,
                    activation,
                    sigma_smoothing,
                    flip_indices,
                    gradients)

# Print the model summary
model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [50]:
model.output

<tf.Tensor 'unet_prediction_25/truediv:0' shape=(None, None, None, None, 33) dtype=float32>

In [53]:
model.output[0]

<tf.Tensor 'strided_slice:0' shape=(None, None, None, 33) dtype=float32>

In [2]:
import numpy as np
import tensorflow as tf
from ext.neuron import models as nrn_models

def extend_model_with_new_classes(original_model_path, num_new_classes=3):
    # Recreate the original model with 33 classes and 1 input channel
    original_model = nrn_models.unet(
        input_shape=[None, None, None, 1],
        nb_labels=4,
        nb_levels=5,
        nb_conv_per_level=2,
        conv_size=3,
        nb_features=24,
        feat_mult=2,
        activation='elu',
        batch_norm=-1
    )
    
    # Load original weights
    original_model.load_weights(original_model_path, by_name=True)
    
    # Recreate the extended model with 36 classes and 4 input channels
    extended_model = nrn_models.unet(
        input_shape=[None, None, None, 4],
        nb_labels=4,  # Decreased to 4
        nb_levels=5,
        nb_conv_per_level=2,
        conv_size=3,
        nb_features=24,
        feat_mult=2,
        activation='elu',
        batch_norm=-1
    )
    
    # Copy weights for existing 33 classes and initialize new weights
    for i, layer in enumerate(extended_model.layers):
        original_layer = original_model.layers[i] if i < len(original_model.layers) else None
        
        # Copy weights for existing layers
        if original_layer and layer.name == original_layer.name:
            weights = original_layer.get_weights()
            
            # Special handling for convolutional layers
            if 'conv' in layer.name and len(weights) == 2:  # Conv layers have kernel and bias
                orig_kernel, orig_bias = weights
                
                # Adjust kernel for 4 input channels
                # Original kernel shape: [kernel_size, kernel_size, 1, filters]
                # Extended kernel shape: [kernel_size, kernel_size, 4, filters]
                extended_kernel = np.tile(orig_kernel, (1, 1, 4, 1)) / 4  # Divide by 4 for initialization
                
                # Bias remains the same
                extended_bias = orig_bias
                
                layer.set_weights([extended_kernel, extended_bias])
            
            # Special handling for the likelihood layer
            elif 'unet_likelihood' in layer.name:
                orig_kernel, orig_bias = weights
                
                # Extend kernel for new classes
                extended_kernel = np.random.normal(
                    0, 0.01, (1, 1, 1, 24, 36)
                ).astype(orig_kernel.dtype)
                extended_kernel[:, :, :, :, :33] = orig_kernel
                
                # Extend bias
                extended_bias = np.zeros(36, dtype=orig_bias.dtype)
                extended_bias[:33] = orig_bias
                
                layer.set_weights([extended_kernel, extended_bias])
            
            # For other layers, if they exist in both models, copy weights directly
            elif weights:
                layer.set_weights(weights)
    
    return extended_model

# Usage
model_path = "models/only_brats_synthseg_2.0.h5"
extended_model = extend_model_with_new_classes(model_path)

# Verify the new model
print("Extended Model Summary:")
extended_model.summary()

# Optional: Save the new model
extended_model.save("models/4channels_brats_synthseg_2.0.h5")

ValueError: Layer weight shape (3, 3, 3, 24, 24) not compatible with provided weight shape (3, 3, 3, 96, 24)

In [45]:
import h5py
import numpy as np
import tensorflow as tf
from ext.neuron import models as nrn_models

def diagnose_model_labels(model_path):
    # Open the H5 file to inspect contents
    with h5py.File(model_path, 'r') as f:
        print("H5 File Structure:")
        def print_structure(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"Group: {name}")
            elif isinstance(obj, h5py.Dataset):
                try:
                    print(f"Dataset: {name}, Shape: {obj.shape}, Dtype: {obj.dtype}")
                except Exception as e:
                    print(f"Dataset: {name}, Error reading details: {e}")
        
        f.visititems(print_structure)
    
    # Try to guess potential label configurations
    possible_labels = [
        list(range(55)),  # 0-54
        list(range(1, 56)),  # 1-55
        None  # Will be determined dynamically
    ]
    
    def attempt_model_creation(labels):
        try:
            print(f"\nAttempting model creation with {len(labels) if labels else 'dynamic'} labels")
            
            # Flexible input shape for 3D volume
            input_shape = [None, None, None, 1]
            
            # Try creating the model with different label configurations
            net = nrn_models.unet(
                input_shape=input_shape,
                nb_labels=len(labels) if labels else 55,
                nb_levels=5,
                nb_conv_per_level=2,
                conv_size=3,
                nb_features=24,
                feat_mult=2,
                activation='elu',
                batch_norm=-1
            )
            
            # Print output layer details
            output_layer = net.layers[-1]
            print(f"Output Layer: {output_layer.name}")
            print(f"Output Layer Weights: {[w.shape for w in output_layer.weights]}")
            
            # Attempt weight loading
            try:
                net.load_weights(model_path, by_name=True, skip_mismatch=False)
                print("Weights loaded successfully!")
                return net
            except Exception as e:
                print(f"Weight loading failed: {e}")
                return None
        
        except Exception as e:
            print(f"Model creation failed: {e}")
            return None
    
    # Try different label configurations
    for labels in possible_labels:
        model = attempt_model_creation(labels)
        if model:
            # If successful, print model summary
            model.summary()
            break

# Replace with your actual model path
model_path = "models/synthseg_2.0.h5"
diagnose_model_labels(model_path)

H5 File Structure:
Group: unet_bn_down_0
Group: unet_bn_down_0/unet_bn_down_0
Dataset: unet_bn_down_0/unet_bn_down_0/beta:0, Shape: (24,), Dtype: float32
Dataset: unet_bn_down_0/unet_bn_down_0/gamma:0, Shape: (24,), Dtype: float32
Dataset: unet_bn_down_0/unet_bn_down_0/moving_mean:0, Shape: (24,), Dtype: float32
Dataset: unet_bn_down_0/unet_bn_down_0/moving_variance:0, Shape: (24,), Dtype: float32
Group: unet_bn_down_1
Group: unet_bn_down_1/unet_bn_down_1
Dataset: unet_bn_down_1/unet_bn_down_1/beta:0, Shape: (48,), Dtype: float32
Dataset: unet_bn_down_1/unet_bn_down_1/gamma:0, Shape: (48,), Dtype: float32
Dataset: unet_bn_down_1/unet_bn_down_1/moving_mean:0, Shape: (48,), Dtype: float32
Dataset: unet_bn_down_1/unet_bn_down_1/moving_variance:0, Shape: (48,), Dtype: float32
Group: unet_bn_down_2
Group: unet_bn_down_2/unet_bn_down_2
Dataset: unet_bn_down_2/unet_bn_down_2/beta:0, Shape: (96,), Dtype: float32
Dataset: unet_bn_down_2/unet_bn_down_2/gamma:0, Shape: (96,), Dtype: float32
Datas

In [44]:
model_path = "models/synthseg_2.0.h5" # Replace with the actual path

# Define input parameters
input_shape = [None,None,None,1]  # 3D input with 1 channel
labels_segmentation = given_labels
diagnose_weight_loading(model_path, input_shape, labels_segmentation)

Input Shape: [None, None, None, 1]
Number of Labels: 55

H5 File Keys:
unet_bn_down_0
unet_bn_down_1
unet_bn_down_2
unet_bn_down_3
unet_bn_down_4
unet_bn_up_0
unet_bn_up_1
unet_bn_up_2
unet_bn_up_3
unet_conv_downarm_0_0
unet_conv_downarm_0_1
unet_conv_downarm_1_0
unet_conv_downarm_1_1
unet_conv_downarm_2_0
unet_conv_downarm_2_1
unet_conv_downarm_3_0
unet_conv_downarm_3_1
unet_conv_downarm_4_0
unet_conv_downarm_4_1
unet_conv_uparm_5_0
unet_conv_uparm_5_1
unet_conv_uparm_6_0
unet_conv_uparm_6_1
unet_conv_uparm_7_0
unet_conv_uparm_7_1
unet_conv_uparm_8_0
unet_conv_uparm_8_1
unet_input
unet_likelihood
unet_maxpool_0
unet_maxpool_1
unet_maxpool_2
unet_maxpool_3
unet_merge_5
unet_merge_6
unet_merge_7
unet_merge_8
unet_prediction
unet_up_5
unet_up_6
unet_up_7
unet_up_8

Attempting to load weights:
Layer: unet_input
Layer weights shapes: []
Layer: unet_conv_downarm_0_0
Layer weights shapes: [TensorShape([3, 3, 3, 1, 24]), TensorShape([24])]
Layer: unet_conv_downarm_0_1
Layer weights shapes: [T

In [54]:
model_path = "models/synthseg_2.0.h5"  # Replace with the actual path

with h5py.File(model_path, 'r') as f:
    print("Keys in the file:", list(f.keys()))

Keys in the file: ['unet_bn_down_0', 'unet_bn_down_1', 'unet_bn_down_2', 'unet_bn_down_3', 'unet_bn_down_4', 'unet_bn_up_0', 'unet_bn_up_1', 'unet_bn_up_2', 'unet_bn_up_3', 'unet_conv_downarm_0_0', 'unet_conv_downarm_0_1', 'unet_conv_downarm_1_0', 'unet_conv_downarm_1_1', 'unet_conv_downarm_2_0', 'unet_conv_downarm_2_1', 'unet_conv_downarm_3_0', 'unet_conv_downarm_3_1', 'unet_conv_downarm_4_0', 'unet_conv_downarm_4_1', 'unet_conv_uparm_5_0', 'unet_conv_uparm_5_1', 'unet_conv_uparm_6_0', 'unet_conv_uparm_6_1', 'unet_conv_uparm_7_0', 'unet_conv_uparm_7_1', 'unet_conv_uparm_8_0', 'unet_conv_uparm_8_1', 'unet_input', 'unet_likelihood', 'unet_maxpool_0', 'unet_maxpool_1', 'unet_maxpool_2', 'unet_maxpool_3', 'unet_merge_5', 'unet_merge_6', 'unet_merge_7', 'unet_merge_8', 'unet_prediction', 'unet_up_5', 'unet_up_6', 'unet_up_7', 'unet_up_8']


In [40]:
import h5py
import tensorflow as tf
import keras

def extract_model_architecture(h5_file_path):
    """
    Multiple approaches to extract model architecture from an H5 file
    """
    # Approach 1: Using h5py to inspect file contents
    def h5py_inspect(h5_file_path):
        with h5py.File(h5_file_path, 'r') as f:
            print("H5 File Keys:", list(f.keys()))
            
            # Look for model configuration keys
            if 'model_config' in f.keys():
                config = f['model_config'][()]
                print("Raw Model Configuration:", config)
    
    # Approach 2: Keras model loading
    def keras_model_load(h5_file_path):
        try:
            model = tf.keras.models.load_model(h5_file_path)
            print("Model Summary:")
            model.summary()
            
            # Get model configuration as JSON
            config = model.get_config()
            print("\nModel Configuration:\n", config)
        except Exception as e:
            print(f"Could not load model directly: {e}")
    
    # Approach 3: Manual H5 exploration
    def manual_h5_exploration(h5_file_path):
        with h5py.File(h5_file_path, 'r') as f:
            def print_attrs(name, obj):
                if hasattr(obj, 'attrs'):
                    print(f"{name} Attributes:")
                    for key, value in obj.attrs.items():
                        print(f"  {key}: {value}")
            
            f.visititems(print_attrs)
    
    print("Approach 1: H5py Inspection")
    h5py_inspect(model_path)
    
    print("\nApproach 2: Keras Model Loading")
    keras_model_load(model_path)
    
    print("\nApproach 3: Manual H5 Exploration")
    manual_h5_exploration(model_path)

# Example usage
# extract_model_architecture('path/to/your/model.h5')

In [36]:
data

<Closed HDF5 group>